# 430 — Anatomy mapping (where do the qualifiers cluster?)

Takes the qualifying contacts from `420` and asks **where they sit** and **how tightly they
cluster** anatomically — comparing the **boxcar** vs **Gaussian** windows throughout.

Three anatomical frameworks:
- **Yeo-7 & Yeo-17** functional networks — precomputed in the coords CSVs (no extra cost).
- **Desikan-Killiany gyri** (`aparc`) — built once via MNE + fsaverage (`ensure_aparc_cache`).
- **fsaverage surface renders** — qualifying contacts drawn on the pial surface, red(+)/blue(−).

Summaries reuse the clustering anatomy engine: per-zone **purity / entropy** (how
region-coherent) and **spatial compactness** (mm spread, hemisphere-mirrored).

> ⚙️ **Requires `mne` + `pyvista` + the fsaverage surfaces** (server-side). The first aparc build
> downloads ~50 MB and takes ~30 s; afterwards it is cached. The render cell needs off-screen
> PyVista (set below, before importing pyvista).


In [ ]:
import os
os.environ.setdefault('PYVISTA_OFF_SCREEN', 'true')
os.environ.setdefault('MPLBACKEND', 'Agg')

import os, sys
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown, Image
sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_pool as P

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('../01_FBM_Analysis/outputs/04_ersp_LM_RAWONLY').resolve()

print('INPUT_DIR :', INPUT_DIR, '| exists:', INPUT_DIR.exists())
print('OUTPUTS   :', P.OUTPUTS_ROOT)
print('COORDS    :', P.COORDS_DIR, '| exists:', P.COORDS_DIR.exists())
print('conditions:', P.CONDITIONS, '| zones:', P.DEFAULT_ZONES)
print('feature sets:', P.FEATURE_SETS, '| window shapes:', P.WINDOW_SHAPES)


In [ ]:
USE_DS = True                              # must match what you ran in 420
GRID   = 'ds' if USE_DS else 'full'
df_pool = P.load_pool_table(grid=GRID)
aparc   = P.ensure_aparc_cache()          # builds once (MNE), then cached
coords  = P.load_coords()
print('grid:', GRID, '| pool rows:', len(df_pool), '| aparc contacts:', len(aparc),
      '| coords contacts:', len(coords))


## 1 — Attach anatomy to the qualifiers
Join each qualifying contact (on `patient_id` + normalized contact name) to its Yeo-7, Yeo-17,
Desikan-Killiany gyrus and fsaverage xyz. Unmatched counts are reported (naming differs across
GVA/PAT vs BERN/EL cohorts).


In [ ]:
dfq = df_pool[df_pool.qualifies].copy()
df_q7  = P.attach_anatomy(dfq, coords, aparc, n_networks=7)
df_q17 = P.attach_anatomy(dfq, coords, aparc, n_networks=17)
df_q7[['patient_id', 'contact_norm', 'condition', 'zone', 'window_shape', 'sign',
       'yeo_label', 'aparc_label']].drop_duplicates().head(20)


## 2 — Purity & spatial compactness per zone × window shape
Each **zone** is one "cluster". For both window shapes we score region purity/entropy and
spatial compactness, so boxcar vs Gaussian are directly comparable. (`N_PERM_ANAT > 0` adds the
entropy permutation p — is the zone more region-coherent than chance.)


In [ ]:
N_PERM_ANAT = 1000
for n_net, dfq_net in ((7, df_q7), (17, df_q17)):
    for shape in P.WINDOW_SHAPES:
        sub = dfq_net[dfq_net.window_shape == shape]
        run_dir = P.new_run_dir('anatomy', f'yeo{n_net}', shape)
        print(f'\n=== Yeo-{n_net} · {shape} -> {run_dir} ===')
        anat, comp, id2name = P.summarize_anatomy(sub, aparc, coords, run_dir,
                                                  by='zone', n_perm=N_PERM_ANAT)
        display(anat); display(comp)


## 3 — fsaverage surface renders
Qualifying contacts of each condition × zone on the pial surface (red = +, blue = −), six views.
Uses the **boxcar** window by default — switch `RENDER_SHAPE` to compare.


In [ ]:
RENDER_SHAPE = 'boxcar'
ren = P.new_run_dir('anatomy', 'renders')
zones = list(df_q7.zone.dropna().unique())
for cond in P.CONDITIONS:
    for zone in zones:
        pngs = P.render_zone_brains(df_q7, ren, condition=cond, zone=zone,
                                    window_shape=RENDER_SHAPE)
        for p in pngs[:2]:           # show 2 views inline to keep the notebook light
            display(Image(filename=str(p)))
